
---

### **Step 1 – What is PaDEL-Descriptor**

**PaDEL-Descriptor** is an open-source software that calculates over **1,875 molecular descriptors** and **12 types of fingerprints** (e.g., PubChem, ECFP) to translate chemical structures into **numerical, machine-readable data** for computational modeling.

It is a **Java-based tool** (developed by **Yap Chun Wei**) designed to calculate **1D, 2D, and 3D descriptors** and fingerprints.
It can calculate **hundreds of 1D/2D descriptors**, **over 100 3D descriptors**, and **12+ fingerprint types** including **MACCS, PubChem, and ECFP-like circular fingerprints**.

It is widely used in **drug discovery** to convert molecular structures (often from **SMILES notation**) into numerical formats for **machine learning models** that predict biological activity.

---

### **Step 2 – What are Molecular Descriptors**

**Molecular descriptors** are **quantitative representations of molecular properties** (physicochemical, topological, and 3D) derived from chemical structure.
They are the final result of a **logical and mathematical transformation** of chemical information into a **useful number**.

They include:

* **0D/1D/2D descriptors** such as atomic composition, molecular weight, formula, and topological indices.
* **3D descriptors** such as molecular shape, electrostatic potential, and geometrical properties (e.g., Polar Surface Area).

These descriptors are used in **QSAR, virtual screening, and ADME/T prediction** to link molecular structure with properties like **solubility, toxicity, and bioactivity**.

---

### **Step 3 – Relationship Between PaDEL and Molecular Descriptors**

**PaDEL-Descriptor** is the **software tool**.
**Molecular descriptors** are the **numerical values** calculated by software like PaDEL.

PaDEL is particularly favored because it produces a **large and diverse set of descriptors and fingerprints**, making it highly effective for **virtual screening and computational drug discovery**.


______________________________________________________

**Code 1: **The following code is simply installing software tools that the rest of your chemistry code needs.

**pip** is a tool used to download and install Python libraries from the internet.

In [1]:
!pip install -q padelpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 78.0 MB/s eta 0:00:00


In [8]:
import pandas as pd
df3 = pd.read_csv('acetylcholinesterase_04_bioactivity_data_3class_pIC50_short.csv')
df3.head()

,Unnamed: 0,molecule_chembl_id,canonical_smiles,class,MW,LogP,NumHDonors,NumHAcceptors,pIC50
0,0,CHEMBL133897,CCOc1nn(-c2cccc(OCc3ccccc3)c2)c(=O)o1,active,312.325,2.8032,0,6,6.124939
1,1,CHEMBL336398,O=C(N1CCCCC1)n1nc(-c2ccc(Cl)cc2)nc1SCC1CC1,active,376.913,4.5546,0,5,7.000000
2,2,CHEMBL131588,CN(C(=O)n1nc(-c2ccc(Cl)cc2)nc1SCC(F)(F)F)c1ccccc1,inactive,426.851,5.3574,0,5,4.301030
3,3,CHEMBL130628,O=C(N1CCCCC1)n1nc(-c2ccc(Cl)cc2)nc1SCC(F)(F)F,active,404.845,4.7069,0,5,6.522879
4,4,CHEMBL130478,CSc1nc(-c2ccc(OC(F)(F)F)cc2)nn1C(=O)N(C)C,active,346.334,3.0953,0,6,6.096910


______________________________________________________

**This code takes a table of molecule data and prepares a simple text file from it, then tells you how many molecules are inside that file.

** This line does the following:

**df3** is a table (like an Excel sheet) that contains many columns about molecules.

From this table, it selects only two columns:

**canonical_smiles** → the chemical structure of the molecule (written as text)

**molecule_chembl_id** → the ID number of that molecule

These two columns are saved into a new file called: molecule.smi

In [4]:
df3[['canonical_smiles','molecule_chembl_id']].to_csv('molecule.smi', sep='\t', index=False, header=False)
print('Total molecules:', sum(1 for _ in open('molecule.smi')))

Total molecules: 19


___________________________________________________

**The code below reads all molecules from the file and stores their structures and IDs so they can be analyzed by drug-discovery software.**

In [5]:
from padelpy import from_smiles
smiles=[]; names=[]
with open('molecule.smi') as f:
    for line in f:
        if line.strip():
            s,n = line.strip().split()
            smiles.append(s); names.append(n)
print('Loaded', len(smiles), 'molecules')

Loaded 19 molecules


**This code converts chemical structures into numbers that a computer (or AI model) can understand.**

**It uses a chemistry tool called PaDEL to calculate molecular fingerprints for each molecule.**

In [7]:
print("Running PaDEL… this may take several minutes")

from padelpy import from_smiles
import pandas as pd

data = from_smiles(
    smiles,
    fingerprints=True,
    descriptors=False,
    timeout=600
)

# Convert output to DataFrame
df_fp = pd.DataFrame(data)

# Add molecule names
df_fp.insert(0, "Name", names)

print("Descriptor matrix shape:", df_fp.shape)
df_fp.head()


Running PaDEL… this may take several minutes
Descriptor matrix shape: (19, 882)


,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL133897,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL336398,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL131588,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL130628,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL130478,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
df_fp.to_csv('descriptors_output.csv', index=False)
print('Saved descriptors_output.csv')

Saved descriptors_output.csv


In [8]:
df3_X = pd.read_csv('descriptors_output.csv').drop(columns=['Name'])
df3_Y = df3['pIC50']
dataset3 = pd.concat([df3_X, df3_Y], axis=1)
dataset3.to_csv('acetylcholinesterase_06_bioactivity_data_3class_pIC50_pubchem_fp.csv', index=False)
print('Final dataset saved')

Final dataset saved
